# ATP RANKING

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("silver_atp_rankings").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [11]:
# tb_atp_rankings = (
#     spark.read
#     .format("jdbc")
#     .option("url", os.getenv("JDBC_URL"))
#     .option("dbtable", "bronze.tb_atp_rankings")
#     .option("user", os.getenv("DB_USER"))
#     .option("password", os.getenv("DB_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
#     )


tb_atp_rankings = spark.read.csv("../../data/bronze/tb_atp_rankings.csv", sep=',', header=True)

## Ranking

In [14]:
df = (
    tb_atp_rankings
    .select(
        f.first("date_week").alias("DATE_WEEK_RANKING"),
        f.first("rank").alias("NUM_PLAYER_RANK"),
        f.first("player").alias("DES_PLAYER_NAME"),
        f.first("age").alias("NUM_PLAYER_AGE"),
        f.first("official_points").alias("NUM_PLAYER_RANK_PTS"),
        f.first("lost_earned_points").alias("NUM_PLAYER_LE_PTS"),
        f.first("tourn_played").alias("NUM_PLAYER_TOURNEY_PLAYED"),
        f.first("dropping").alias("NUM_PLAYER_DROP_PTS"),
        f.first("next_best").alias("NUM_PLAYER_NEXT_BEST"),

        f.first("DATE_INGESTION").alias("DATE_INGESTION")
    )
)

## Save dataframe

### Local

In [16]:
df.toPandas().to_csv(
    r"../../data/silver/tb_atp_rankings.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [17]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_rankings")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)